In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
import matplotlib.pyplot as plt

# ml
from sklearn.model_selection import train_test_split
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize
from sklearn.linear_model import Lasso
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LassoCV

bf MICE

In [2]:
# data
file_path = r"/Users/selina/Desktop/365mc_re/0.code/1_preprocessing/c_df_raw.csv" # merge_df
df = pd.read_csv(file_path) # 7046

print(df.columns.tolist())

['ID', 'branch', 'doctor', 'p_date', 'body_code', 'body_name', 'edema', 'sex', 'age', 'height', 'fat_cc', 'bf_weight', 'af_weight', 'bf_size', 'af_size', 'bf_tbw', 'af_tbw', 'bf_protein', 'af_protein', 'bf_mineral', 'af_mineral', 'bf_ffm', 'af_ffm', 'bf_smm', 'af_smm', 'bf_bfm', 'af_bfm', 'bf_whr', 'af_whr', 'Liposuction type']


In [3]:
unique_ids = df['ID'].unique()
print(unique_ids)
print(f"total p, unique: {len(unique_ids)}") # 4947

[360073593 180033405 360016942 ... 350009171 180031961 320005499]
total p, unique: 4947


In [4]:
df # 8064

,ID,branch,doctor,p_date,body_code,body_name,edema,sex,age,height,...,af_mineral,bf_ffm,af_ffm,bf_smm,af_smm,bf_bfm,af_bfm,bf_whr,af_whr,Liposuction type
0,360073593,서울,pingwa,20240129,7,복부,약간의 붓기,F,38,166.0,...,2.98,41.8,39.7,22.2,21.0,24.1,22.1,0.93,0.94,surgery
1,180033405,대전,ssungm,20240125,7,복부,NaN,F,44,163.0,...,2.70,38.6,42.3,21.0,23.1,16.6,16.0,0.93,0.94,surgery
2,360016942,서울,pingwa,20240110,7,복부,NaN,F,44,154.0,...,2.76,38.0,38.5,20.4,20.6,23.6,21.2,0.91,0.87,surgery
3,360073502,서울,pingwa,20240124,6,팔,약간의 붓기,F,42,160.0,...,3.13,41.7,43.8,22.7,24.0,30.3,26.4,0.99,0.95,surgery
4,360073596,서울,pingwa,20240111,12,허벅지,약간의 붓기,F,21,165.0,...,3.46,48.2,46.4,26.4,25.3,34.1,33.5,0.88,0.86,surgery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8059,320005499,부천,md2217,20240110,12,허벅지,NaN,F,36,162.0,...,NaN,47.6,NaN,26.4,NaN,34.9,NaN,1.00,NaN,lams
8060,60022070,영등포,md2217,20240308,12,허벅지,약간의 붓기,F,33,173.0,...,NaN,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams
8061,60022070,영등포,md2217,20240308,12,허벅지,약간의 붓기,F,33,173.0,...,NaN,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams
8062,60022070,영등포,md2217,20240308,12,허벅지,약간의 붓기,F,33,173.0,...,NaN,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams


In [5]:
# BMI: kg/m² = weight / (height in m)^2
df['BMI'] = df['bf_weight'] / ((df['height'] / 100) ** 2)
df['BMI'] = df['BMI'].round(2)
df

,ID,branch,doctor,p_date,body_code,body_name,edema,sex,age,height,...,bf_ffm,af_ffm,bf_smm,af_smm,bf_bfm,af_bfm,bf_whr,af_whr,Liposuction type,BMI
0,360073593,서울,pingwa,20240129,7,복부,약간의 붓기,F,38,166.0,...,41.8,39.7,22.2,21.0,24.1,22.1,0.93,0.94,surgery,23.91
1,180033405,대전,ssungm,20240125,7,복부,NaN,F,44,163.0,...,38.6,42.3,21.0,23.1,16.6,16.0,0.93,0.94,surgery,20.78
2,360016942,서울,pingwa,20240110,7,복부,NaN,F,44,154.0,...,38.0,38.5,20.4,20.6,23.6,21.2,0.91,0.87,surgery,25.97
3,360073502,서울,pingwa,20240124,6,팔,약간의 붓기,F,42,160.0,...,41.7,43.8,22.7,24.0,30.3,26.4,0.99,0.95,surgery,28.12
4,360073596,서울,pingwa,20240111,12,허벅지,약간의 붓기,F,21,165.0,...,48.2,46.4,26.4,25.3,34.1,33.5,0.88,0.86,surgery,30.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8059,320005499,부천,md2217,20240110,12,허벅지,NaN,F,36,162.0,...,47.6,NaN,26.4,NaN,34.9,NaN,1.00,NaN,lams,31.44
8060,60022070,영등포,md2217,20240308,12,허벅지,약간의 붓기,F,33,173.0,...,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams,33.41
8061,60022070,영등포,md2217,20240308,12,허벅지,약간의 붓기,F,33,173.0,...,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams,33.41
8062,60022070,영등포,md2217,20240308,12,허벅지,약간의 붓기,F,33,173.0,...,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams,33.41


Define ID based on the number of surgeries

In [6]:
# missing (p_date)
missing_count = df['p_date'].isna().sum()
missing_ratio = df['p_date'].isna().mean() * 100

print(f"결측치 개수: {missing_count}")
print(f"결측치 비율: {missing_ratio:.2f}%")

결측치 개수: 0
결측치 비율: 0.00%


In [7]:
df2 = df[df['p_date'].notna()].copy()
print(f"결측치 제거 후 남은 행 수: {len(df2)}") # 8064

결측치 제거 후 남은 행 수: 8064


In [ ]:
# rename_dict = {
#     'psentry': 'ID',
#     '지점명': 'branch',  # label encoding 
#     'DOCTOR_NAME': 'doctor',  # label encoding
#     'p_date': 'p_date', # non included
#     'body_code': 'body_code', # num 9 > 7
#     'body_name': 'body_name', # rename
#     '부종(현재 붓기가 있나요?)': 'edema',  # ordinary encoding > MICE
#     'sex': 'sex', # label encoding
#     'age': 'age', # outlier check
#     '키': 'height', # outlier check
#     '추출량(Bottle)': 'fat_cc', # outlier check
#     '인바디체중': 'bf_weight', # outlier check
#     '전사이즈': 'bf_size', # outlier check
#     '추출량(Bottle)': 'fat_cc', # outlier check
#     '체수분': 'bf_tbw', # outlier check
#     '단백질': 'bf_protein', # outlier check
#     '무기질': 'bf_mineral', # outlier check 
#     '제지방량': 'bf_ffm', # outlier check
#     '골격근량': 'bf_smm', # outlier check
#     '체지방량': 'bf_bfm', # outlier check
#     '복부지방률': 'bf_whr', # outlier check
# }

In [8]:
# body
df2.groupby(['body_code', 'body_name']).size().reset_index(name='count')

,body_code,body_name,count
0,0,공통,8
1,4,등,344
2,6,팔,3551
3,7,복부,1948
4,8,힙,25
5,9,힙업,104
6,10,러브핸들,566
7,12,허벅지,1419
8,13,종아리,99


In [9]:
# 1. '공통' 제거
df3 = df2[df2['body_name'] != '공통'].copy() # -8
# 2. new_mapping
new_code_map = {
    7: 1,  # 복부 → Abdomen
    6: 2,  # 팔 → Arms
    4: 3,  # 등 → Backs
    8: 4, 9: 4,  # 힙, 힙업 → Buttocks
    13: 5,  # 종아리 → Calves
    10: 6,  # 러브핸들 → Flanks
    12: 7   # 허벅지 → Thighs
}
new_name_map = {
    1: 'Abdomen',
    2: 'Arms',
    3: 'Backs',
    4: 'Buttocks',
    5: 'Calves',
    6: 'Flanks',
    7: 'Thighs'
}

df3['body_code'] = df3['body_code'].map(new_code_map)
df3['body_name'] = df3['body_code'].map(new_name_map)
print(df3[['body_code', 'body_name']].drop_duplicates())  # 8056

      body_code body_name
0             1   Abdomen
3             2      Arms
4             7    Thighs
14            5    Calves
1027          6    Flanks
1031          3     Backs
1115          4  Buttocks


In [10]:
# body
df3.groupby(['body_code', 'body_name']).size().reset_index(name='count')

,body_code,body_name,count
0,1,Abdomen,1948
1,2,Arms,3551
2,3,Backs,344
3,4,Buttocks,129
4,5,Calves,99
5,6,Flanks,566
6,7,Thighs,1419


In [11]:
# sex and edema (bf MICE)
print(df3['edema'].value_counts(dropna=False)) # 마지막에 MICE 예정
print(df3['sex'].value_counts(dropna=False))

edema
약간의 붓기             5138
NaN                2038
생활 불편이 있는 심한 붓기     445
없음                  435
Name: count, dtype: int64
sex
F    7855
M     198
E       3
Name: count, dtype: int64


In [12]:
# sex: encoding 
df4 = df3[df3['sex'].isin(['F', 'M'])].copy()
sex_map = {'M': 1, 'F': 2}
df4['sex'] = df4['sex'].map(sex_map)

# edema: ordinary encoding (label encoding X)
edema_map = {
    '없음': 1,
    '약간의 붓기': 2,
    '생활 불편이 있는 심한 붓기': 3,
    np.nan: 1
}
df4['edema'] = df4['edema'].map(edema_map).fillna(0).astype(int)

print(df4['sex'].value_counts())
print(df4['edema'].value_counts())

sex
2    7855
1     198
Name: count, dtype: int64
edema
2    5138
1    2470
3     445
Name: count, dtype: int64


In [13]:
df4

,ID,branch,doctor,p_date,body_code,body_name,edema,sex,age,height,...,bf_ffm,af_ffm,bf_smm,af_smm,bf_bfm,af_bfm,bf_whr,af_whr,Liposuction type,BMI
0,360073593,서울,pingwa,20240129,1,Abdomen,2,2,38,166.0,...,41.8,39.7,22.2,21.0,24.1,22.1,0.93,0.94,surgery,23.91
1,180033405,대전,ssungm,20240125,1,Abdomen,1,2,44,163.0,...,38.6,42.3,21.0,23.1,16.6,16.0,0.93,0.94,surgery,20.78
2,360016942,서울,pingwa,20240110,1,Abdomen,1,2,44,154.0,...,38.0,38.5,20.4,20.6,23.6,21.2,0.91,0.87,surgery,25.97
3,360073502,서울,pingwa,20240124,2,Arms,2,2,42,160.0,...,41.7,43.8,22.7,24.0,30.3,26.4,0.99,0.95,surgery,28.12
4,360073596,서울,pingwa,20240111,7,Thighs,2,2,21,165.0,...,48.2,46.4,26.4,25.3,34.1,33.5,0.88,0.86,surgery,30.23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8059,320005499,부천,md2217,20240110,7,Thighs,1,2,36,162.0,...,47.6,NaN,26.4,NaN,34.9,NaN,1.00,NaN,lams,31.44
8060,60022070,영등포,md2217,20240308,7,Thighs,2,2,33,173.0,...,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams,33.41
8061,60022070,영등포,md2217,20240308,7,Thighs,2,2,33,173.0,...,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams,33.41
8062,60022070,영등포,md2217,20240308,7,Thighs,2,2,33,173.0,...,50.4,NaN,27.6,NaN,49.6,NaN,1.07,NaN,lams,33.41


In [14]:
# label encoding
from sklearn.preprocessing import LabelEncoder
df5 = df4.copy()
label_cols = {
    'branch': 'branch_',
    'doctor': 'doctor_',
    'Liposuction type': 'Liposuction_type_'
}
for col, new_col in label_cols.items():
    le = LabelEncoder()
    df5[new_col] = le.fit_transform(df5[col].astype(str)) 
    
print(df5[['branch', 'branch_']].drop_duplicates().sort_values('branch_'))
print(df5[['doctor', 'doctor_']].drop_duplicates().sort_values('doctor_')) # brach:20, doctor:38
print(df5[['Liposuction type', 'Liposuction_type_']].drop_duplicates().sort_values('Liposuction_type_')) # lams/surgical:1

     branch  branch_
1055   강남본점        0
1792     구리        1
1042     노원        2
10       대구        3
1        대전        4
5        부산        5
1083     부천        6
1021     분당        7
0        서울        8
1078   성신여대        9
1029     수원       10
1115     신촌       11
1022   안양평촌       12
1045    영등포       13
21       인천       14
1032     일산       15
1050     천안       16
1048     천호       17
1054     청주       18
1096    해운대       19
      doctor  doctor_
3492  04kjh1        0
1042   04ldk        1
2156   04lgh        2
96     04lsy        3
5       21sc        4
240    34ajh        5
4006    34hs        6
198   34k122        7
131   34skim        8
51      34sw        9
1264    35hz       10
99    35kmhh       11
802   36ssjw       12
1063   52eyp       13
1055   52nat       14
276   JJM365       15
1459    buty       16
1894     ckh       17
1847    drhj       18
1106  hws202       19
1020  jj8181       20
1792  jrang1       21
618     jwju       22
615     kg35       23
40     kh

In [15]:
df6 = df5.rename(columns={
    'sex': 'Sex',  
    'age': 'Age',
    'height': 'Height',
    'bf_weight': 'Weight',    
    'bf_size': 'Size',
    'bf_whr': 'WHR',
    'BMI': 'BMI',
    'Liposuction_type_': 'Liposuction type',   
    'body_code': 'Liposuction site',   
    'fat_cc': 'Fat volume',
    'edema': 'Edema', 
    'bf_protein': 'Body protein',
    'bf_mineral': 'Body mineral',
    'bf_tbw': 'TBW',
    'bf_ffm': 'FFM',
    'bf_smm': 'SMM',
    'bf_bfm': 'BFM'
})

In [16]:
final_365 = df6.copy()
final_365.to_csv('final_365.csv', index=False, encoding='utf-8-sig') # final_365

Data load

In [17]:
final_365 = pd.read_csv('final_365.csv')
final_365

,ID,branch,doctor,p_date,Liposuction site,body_name,Edema,Sex,Age,Height,...,af_smm,BFM,af_bfm,WHR,af_whr,Liposuction type,BMI,branch_,doctor_,Liposuction type.1
0,360073593,서울,pingwa,20240129,1,Abdomen,2,2,38,166.0,...,21.0,24.1,22.1,0.93,0.94,surgery,23.91,8,32,1
1,180033405,대전,ssungm,20240125,1,Abdomen,1,2,44,163.0,...,23.1,16.6,16.0,0.93,0.94,surgery,20.78,4,36,1
2,360016942,서울,pingwa,20240110,1,Abdomen,1,2,44,154.0,...,20.6,23.6,21.2,0.91,0.87,surgery,25.97,8,32,1
3,360073502,서울,pingwa,20240124,2,Arms,2,2,42,160.0,...,24.0,30.3,26.4,0.99,0.95,surgery,28.12,8,32,1
4,360073596,서울,pingwa,20240111,7,Thighs,2,2,21,165.0,...,25.3,34.1,33.5,0.88,0.86,surgery,30.23,8,32,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8048,320005499,부천,md2217,20240110,7,Thighs,1,2,36,162.0,...,NaN,34.9,NaN,1.00,NaN,lams,31.44,6,28,0
8049,60022070,영등포,md2217,20240308,7,Thighs,2,2,33,173.0,...,NaN,49.6,NaN,1.07,NaN,lams,33.41,13,28,0
8050,60022070,영등포,md2217,20240308,7,Thighs,2,2,33,173.0,...,NaN,49.6,NaN,1.07,NaN,lams,33.41,13,28,0
8051,60022070,영등포,md2217,20240308,7,Thighs,2,2,33,173.0,...,NaN,49.6,NaN,1.07,NaN,lams,33.41,13,28,0


In [18]:
final_365.drop(columns=['branch','doctor','p_date','body_name','af_tbw','af_protein','af_mineral','af_ffm','af_smm','af_bfm','af_whr', 'Liposuction type'], inplace=True)

In [19]:
missing_mask = final_365[['af_weight', 'af_size']].isnull().any(axis=1)
print('결측 행 수:', missing_mask.sum())   #249
final_365_ = final_365.dropna(subset=['af_weight', 'af_size'])
final_365_   # 7084

결측 행 수: 249


,ID,Liposuction site,Edema,Sex,Age,Height,Fat volume,Weight,af_weight,Size,...,Body protein,Body mineral,FFM,SMM,BFM,WHR,BMI,branch_,doctor_,Liposuction type.1
0,360073593,1,2,2,38,166.0,600.0,65.9,61.8,94.5,...,8.0,3.10,41.8,22.2,24.1,0.93,23.91,8,32,1
1,180033405,1,1,2,44,163.0,1000.0,55.2,58.3,84.0,...,7.6,2.66,38.6,21.0,16.6,0.93,20.78,4,36,1
2,360016942,1,1,2,44,154.0,750.0,61.6,59.7,89.0,...,7.4,2.65,38.0,20.4,23.6,0.91,25.97,8,32,1
3,360073502,2,2,2,42,160.0,1300.0,72.0,70.2,34.0,...,8.2,3.00,41.7,22.7,30.3,0.99,28.12,8,32,1
4,360073596,7,2,2,21,165.0,2100.0,82.3,79.9,71.5,...,9.4,3.57,48.2,26.4,34.1,0.88,30.23,8,32,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7853,350011246,7,2,2,29,163.0,1200.0,56.5,54.4,57.0,...,7.6,2.88,39.5,21.3,17.0,0.81,21.27,3,20,0
7859,520006104,6,1,2,26,157.0,200.0,55.2,55.1,80.0,...,6.7,2.51,34.3,18.1,20.9,0.89,22.39,0,19,0
7864,520006104,1,1,2,26,157.0,350.0,55.2,55.1,80.0,...,6.7,2.51,34.3,18.1,20.9,0.89,22.39,0,19,0
7865,210043890,3,2,2,26,161.0,900.0,72.3,64.4,90.0,...,8.0,2.99,40.9,22.1,31.4,0.89,27.89,5,18,0


In [20]:
final_365_ = final_365_.copy()

final_365_.drop(columns=['branch_', 'doctor_'], inplace=True)
final_365_.rename(columns={'Liposuction type.1': 'Liposuction type'}, inplace=True)
final_365_

,ID,Liposuction site,Edema,Sex,Age,Height,Fat volume,Weight,af_weight,Size,af_size,TBW,Body protein,Body mineral,FFM,SMM,BFM,WHR,BMI,Liposuction type
0,360073593,1,2,2,38,166.0,600.0,65.9,61.8,94.5,77.0,30.7,8.0,3.10,41.8,22.2,24.1,0.93,23.91,1
1,180033405,1,1,2,44,163.0,1000.0,55.2,58.3,84.0,74.0,28.3,7.6,2.66,38.6,21.0,16.6,0.93,20.78,1
2,360016942,1,1,2,44,154.0,750.0,61.6,59.7,89.0,80.0,27.9,7.4,2.65,38.0,20.4,23.6,0.91,25.97,1
3,360073502,2,2,2,42,160.0,1300.0,72.0,70.2,34.0,29.5,30.5,8.2,3.00,41.7,22.7,30.3,0.99,28.12,1
4,360073596,7,2,2,21,165.0,2100.0,82.3,79.9,71.5,67.4,35.2,9.4,3.57,48.2,26.4,34.1,0.88,30.23,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7853,350011246,7,2,2,29,163.0,1200.0,56.5,54.4,57.0,55.0,29.0,7.6,2.88,39.5,21.3,17.0,0.81,21.27,0
7859,520006104,6,1,2,26,157.0,200.0,55.2,55.1,80.0,77.7,25.1,6.7,2.51,34.3,18.1,20.9,0.89,22.39,0
7864,520006104,1,1,2,26,157.0,350.0,55.2,55.1,80.0,77.8,25.1,6.7,2.51,34.3,18.1,20.9,0.89,22.39,0
7865,210043890,3,2,2,26,161.0,900.0,72.3,64.4,90.0,86.0,29.9,8.0,2.99,40.9,22.1,31.4,0.89,27.89,0


In [21]:
# 0512 revision
final_365_ = final_365_.drop(columns=['Fat volume', 'Edema']) # pre-surgery

In [22]:
# impuataion은 data leakage 이유로, split 하고 나서 진행해야함.

In [23]:
index=final_365_['ID']
label = final_365_[['af_weight', 'af_size']]

label_n = ['ID','af_weight','af_size']
data = final_365_.drop(label_n, axis=1) 

In [24]:
index

0       360073593
1       180033405
2       360016942
3       360073502
4       360073596
          ...    
7853    350011246
7859    520006104
7864    520006104
7865    210043890
7866    210043890
Name: ID, Length: 7804, dtype: int64

In [25]:
label

,af_weight,af_size
0,61.8,77.0
1,58.3,74.0
2,59.7,80.0
3,70.2,29.5
4,79.9,67.4
...,...,...
7853,54.4,55.0
7859,55.1,77.7
7864,55.1,77.8
7865,64.4,86.0


In [26]:
data

,Liposuction site,Sex,Age,Height,Weight,Size,TBW,Body protein,Body mineral,FFM,SMM,BFM,WHR,BMI,Liposuction type
0,1,2,38,166.0,65.9,94.5,30.7,8.0,3.10,41.8,22.2,24.1,0.93,23.91,1
1,1,2,44,163.0,55.2,84.0,28.3,7.6,2.66,38.6,21.0,16.6,0.93,20.78,1
2,1,2,44,154.0,61.6,89.0,27.9,7.4,2.65,38.0,20.4,23.6,0.91,25.97,1
3,2,2,42,160.0,72.0,34.0,30.5,8.2,3.00,41.7,22.7,30.3,0.99,28.12,1
4,7,2,21,165.0,82.3,71.5,35.2,9.4,3.57,48.2,26.4,34.1,0.88,30.23,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7853,7,2,29,163.0,56.5,57.0,29.0,7.6,2.88,39.5,21.3,17.0,0.81,21.27,0
7859,6,2,26,157.0,55.2,80.0,25.1,6.7,2.51,34.3,18.1,20.9,0.89,22.39,0
7864,1,2,26,157.0,55.2,80.0,25.1,6.7,2.51,34.3,18.1,20.9,0.89,22.39,0
7865,3,2,26,161.0,72.3,90.0,29.9,8.0,2.99,40.9,22.1,31.4,0.89,27.89,0


In [27]:
# impuataion은 data leakage 이유로, split 하고 나서 진행해야함.

In [28]:
# data split
train_x_, test_x_, train_y, test_y = train_test_split(data, label, test_size=0.3, random_state=1)  # test_size default 0.25
# train_x_, test_x_, train_y_, test_y_ = train_test_split(data, label, test_size = 0.2, stratify = label, random_state = 1) only for categorical

In [29]:
train_x_

,Liposuction site,Sex,Age,Height,Weight,Size,TBW,Body protein,Body mineral,FFM,SMM,BFM,WHR,BMI,Liposuction type
7680,2,2,31,163.0,63.4,33.9,29.6,7.9,2.99,40.5,21.9,22.9,0.83,23.86,0
2866,7,2,26,170.0,60.3,61.0,32.0,8.4,3.22,43.6,23.5,16.7,0.81,20.87,0
6443,7,2,33,159.0,51.1,57.0,27.0,7.3,2.54,36.8,19.7,14.3,0.78,20.21,0
6856,1,2,50,160.0,52.8,86.2,25.6,6.7,2.52,34.8,18.2,18.0,0.88,20.62,0
5414,1,2,68,153.0,57.0,86.0,27.9,7.4,2.66,38.0,20.5,19.0,0.92,24.35,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
910,1,2,41,170.0,64.8,85.0,31.6,8.5,3.11,43.2,23.5,21.6,0.88,22.42,1
5238,1,2,42,165.0,57.6,79.5,28.2,7.5,2.76,38.5,20.6,19.1,0.89,21.16,0
4026,7,2,38,155.0,42.6,48.7,25.5,7.0,2.32,34.8,18.9,7.8,0.77,17.73,0
236,2,2,37,164.0,59.8,32.0,32.5,8.9,33.00,44.4,24.6,15.4,0.84,22.23,1


In [30]:
test_x_

,Liposuction site,Sex,Age,Height,Weight,Size,TBW,Body protein,Body mineral,FFM,SMM,BFM,WHR,BMI,Liposuction type
424,2,2,28,152.0,50.1,27.5,23.7,6.3,2.28,32.3,17.1,17.8,0.90,21.68,1
7004,6,2,52,162.0,56.8,75.0,30.9,8.2,2.91,42.0,22.6,14.8,0.89,21.64,0
1600,1,2,42,165.0,NaN,70.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2827,2,2,34,158.0,57.4,30.5,26.7,7.2,2.64,36.5,19.5,20.9,0.82,22.99,0
5965,1,2,31,158.0,60.9,83.0,30.0,8.1,2.90,41.0,22.3,19.9,0.87,24.40,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,1,2,41,158.0,67.4,94.0,29.2,7.9,2.68,39.8,21.7,27.6,0.95,27.00,1
2074,7,2,35,167.0,67.4,61.0,33.0,8.7,3.23,44.9,24.3,22.5,0.88,24.17,0
3741,2,2,34,156.0,60.2,33.5,27.6,7.4,2.65,37.6,20.3,22.6,0.87,24.74,0
3749,2,2,28,162.0,71.5,37.5,28.3,7.4,2.76,38.5,20.5,33.0,0.92,27.24,0


Imputation + winsorization

In [31]:
# missing rate
# max: 30%; MICE
missing_rate = train_x_.isnull().mean() * 100
missing_rate = missing_rate.sort_values(ascending=False)
print(missing_rate) 

BMI                 0.714024
Weight              0.512633
TBW                 0.512633
Body protein        0.512633
Body mineral        0.512633
FFM                 0.512633
SMM                 0.512633
BFM                 0.512633
WHR                 0.512633
Height              0.201391
Liposuction site    0.000000
Sex                 0.000000
Age                 0.000000
Size                0.000000
Liposuction type    0.000000
dtype: float64


In [32]:
# continuous variables with missing values (train_x_)
cols_with_na = [col for col in train_x_.columns if train_x_[col].isna().sum() > 0]

# Imputer (MICE)
imp = IterativeImputer(max_iter=50, random_state=0)  # if 10, warning
train_x_imputed = train_x_.copy()
test_x_imputed = test_x_.copy()

train_x_imputed[cols_with_na] = imp.fit_transform(train_x_[cols_with_na])
test_x_imputed[cols_with_na] = imp.transform(test_x_[cols_with_na])

In [33]:
cols_w = ['Body protein', 'WHR', 'SMM', 'TBW', 'Weight', 'BFM', 'Body mineral', 'FFM', 'Height', 'Age', 'Size', 'BMI']

winsor_check = []
for col in cols_w:
    q01 = train_x_imputed[col].quantile(0.01)
    q99 = train_x_imputed[col].quantile(0.99)
    lower_outliers = (train_x_imputed[col] < q01).sum()
    upper_outliers = (train_x_imputed[col] > q99).sum()
    total_outliers = lower_outliers + upper_outliers

    winsor_check.append({
        'column': col,
        'q01': q01,
        'q99': q99,
        'lower_outliers': lower_outliers,
        'upper_outliers': upper_outliers,
        'total_outliers': total_outliers
    })

winsor_df = pd.DataFrame(winsor_check)

# select cols
apply_cols = winsor_df[winsor_df['total_outliers'] > 10]['column'].tolist()

# winsorization
for col in apply_cols:
    q01 = train_x_imputed[col].quantile(0.01)
    q99 = train_x_imputed[col].quantile(0.99)
    train_x_imputed[col] = train_x_imputed[col].clip(q01, q99)
    test_x_imputed[col] = test_x_imputed[col].clip(q01, q99)

In [34]:
print("Winsorization cols:")
print(apply_cols)

Winsorization cols:
['Body protein', 'WHR', 'SMM', 'TBW', 'Weight', 'BFM', 'Body mineral', 'FFM', 'Height', 'Age', 'Size', 'BMI']


In [35]:
# standardscaler
scaled_cols = ['Body protein', 'WHR', 'SMM', 'TBW', 'Weight', 'BFM', 'Body mineral', 'FFM', 'Height', 'Age', 'Size', 'BMI']

train_x = train_x_imputed.copy()
test_x = test_x_imputed.copy()

scaler = StandardScaler()
scaler.fit(train_x[scaled_cols]) # only for train

train_x[scaled_cols] = scaler.transform(train_x[scaled_cols])
test_x[scaled_cols] = scaler.transform(test_x[scaled_cols])

In [36]:
train_x

,Liposuction site,Sex,Age,Height,Weight,Size,TBW,Body protein,Body mineral,FFM,SMM,BFM,WHR,BMI,Liposuction type
7680,2,2,-0.498445,0.206241,0.032149,-0.931736,-0.130956,-0.144325,-0.258230,-0.111385,-0.125476,0.127280,-0.916407,-0.033660,0
2866,7,2,-1.080675,1.506007,-0.236695,0.116329,0.447466,0.297448,-0.227214,0.436386,0.343827,-0.672705,-1.241142,-0.806471,0
6443,7,2,-0.265553,-0.536482,-1.034553,-0.038367,-0.757580,-0.674453,-0.318912,-0.765177,-0.770768,-0.982377,-1.728245,-0.977058,0
6856,1,2,1.714028,-0.350801,-0.887122,1.090914,-1.094993,-1.204580,-0.321609,-1.118578,-1.210739,-0.504967,-0.104569,-0.871087,0
5414,1,2,2.878488,-1.650567,-0.522883,1.083180,-0.540672,-0.586098,-0.302730,-0.553137,-0.536116,-0.375937,0.544901,0.092988,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
910,1,2,0.666015,1.506007,0.153562,1.044506,0.351063,0.385802,-0.242048,0.365706,0.343827,-0.040459,-0.104569,-0.405850,1
5238,1,2,0.782461,0.577603,-0.470849,0.831799,-0.468369,-0.497744,-0.289245,-0.464786,-0.506785,-0.363034,0.057798,-0.731516,0
4026,7,2,0.316677,-1.279206,-1.497571,-0.359361,-1.119094,-0.939517,-0.348578,-1.118578,-1.005419,-1.472691,-1.890612,-1.458811,0
236,2,2,0.200231,0.391922,-0.280057,-1.005217,0.567971,0.739220,3.788595,0.577747,0.666473,-0.840444,-0.754039,-0.454958,1


Scaler save for streamlt ** (0825)

In [37]:
desired_order = [
    'Sex', 'Age', 'Height', 'Weight', 'Size', 'BMI', 'WHR', 
    'Liposuction type', 'Liposuction site','Body protein',
    'Body mineral', 'TBW', 'FFM', 'SMM', 'BFM'
]

for df in [train_x, test_x]:
    df = df[[col for col in desired_order if col in df.columns]]
df

,Sex,Age,Height,Weight,Size,BMI,WHR,Liposuction type,Liposuction site,Body protein,Body mineral,TBW,FFM,SMM,BFM
424,2,-0.847783,-1.836248,-1.121276,-1.179250,-0.597114,0.220166,1,2,-1.557999,-0.353972,-1.552911,-1.560330,-1.533385,-0.530772
7004,2,1.946920,0.020560,-0.540228,0.657766,-0.607452,0.057798,0,6,0.120738,-0.269017,0.182356,0.153665,0.079844,-0.917862
1600,2,0.782461,0.577603,0.008775,0.479865,-0.220699,0.079728,0,1,-0.011149,-0.015194,0.002433,0.006534,-0.011367,0.006369
2827,2,-0.149107,-0.722163,-0.488193,-1.063228,-0.258524,-1.078774,0,2,-0.762807,-0.305427,-0.829883,-0.818188,-0.829430,-0.130780
5965,2,-0.498445,-0.722163,-0.184661,0.967158,0.105912,-0.266937,0,1,0.032384,-0.270366,-0.034552,-0.023035,-0.008150,-0.259810
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,2,0.666015,-0.722163,0.379043,1.392572,0.777921,1.032004,1,1,-0.144325,-0.300033,-0.227360,-0.235076,-0.184139,0.733720
2074,2,-0.032661,0.948964,0.379043,0.116329,0.046465,-0.104569,0,7,0.562511,-0.225866,0.688476,0.666097,0.578478,0.075668
3741,2,-0.149107,-1.093525,-0.245367,-0.947206,0.193790,-0.266937,0,2,-0.586098,-0.304078,-0.612975,-0.623817,-0.594779,0.088571
3749,2,-0.847783,0.020560,0.734611,-0.792510,0.839953,0.544901,0,2,-0.586098,-0.289245,-0.444268,-0.464786,-0.536116,1.430482


In [38]:
import joblib
bundle = {
    "scaler": scaler,
    "scaled_cols": scaled_cols 
}
joblib.dump(bundle, "scaler_bundle.pkl")
print("Scaler saved")

Scaler saved


Final_model

In [39]:
import sklearn
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor, RegressorChain
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, make_scorer
from scipy import stats
from sklearn.ensemble import VotingRegressor
from sklearn.base import RegressorMixin, BaseEstimator
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [40]:
import xgboost
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)

numpy: 2.2.6
pandas: 2.3.3
scikit-learn: 1.7.2
xgboost: 3.2.0


In [41]:
# RMSE score
def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse_score, greater_is_better=False)

# MAPE with zero handling
def mape_score(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = (y_true != 0) & (~np.isnan(y_true))  # 0과 NaN 값 제외
    if np.sum(mask) == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [42]:
# 95% CI
def calculate_ci(data, confidence=0.95):
    data = np.array(data)
    mean = np.mean(data)
    n = len(data)
    sem = stats.sem(data)
    h = sem * stats.t.ppf((1 + confidence) / 2., n-1)
    return mean, mean - h, mean + h

In [43]:
def regression_cv(model, X, y, cv=5, random_state=42, target_names=None):
    kf = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    metrics = {
        'r2': [], 'mae': [], 'mape': [], 'rmse': [],
        'r2_target': [[] for _ in range(y.shape[1])],
        'mae_target': [[] for _ in range(y.shape[1])],
        'mape_target': [[] for _ in range(y.shape[1])],
        'rmse_target': [[] for _ in range(y.shape[1])]
    }
    feature_importances = []
    feature_names = X.columns if hasattr(X, 'columns') else [f'Feature_{i}' for i in range(X.shape[1])]
    
    if target_names is None:
        target_names = [f'Target_{i}' for i in range(y.shape[1])]
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        fold_r2, fold_mae, fold_mape, fold_rmse = [], [], [], []
        
        for i in range(y.shape[1]):
            y_val_i = y_val.iloc[:, i]
            y_pred_i = y_pred[:, i]
            
            r2_val = r2_score(y_val_i, y_pred_i)
            mae_val = mean_absolute_error(y_val_i, y_pred_i)
            mape_val = mape_score(y_val_i, y_pred_i)
            rmse_val = np.sqrt(mean_squared_error(y_val_i, y_pred_i))
            
            fold_r2.append(r2_val)
            fold_mae.append(mae_val)
            fold_mape.append(mape_val)
            fold_rmse.append(rmse_val)
            
            metrics['r2_target'][i].append(r2_val)
            metrics['mae_target'][i].append(mae_val)
            metrics['mape_target'][i].append(mape_val)
            metrics['rmse_target'][i].append(rmse_val)
        
        metrics['r2'].append(np.mean(fold_r2))
        metrics['mae'].append(np.mean(fold_mae))
        metrics['mape'].append(np.mean(fold_mape))
        metrics['rmse'].append(np.mean(fold_rmse))
        
        fold_imp = []
        if hasattr(model, 'feature_importances_'):
            fold_imp = model.feature_importances_
        elif hasattr(model, 'estimators_'):
            imps = [est.feature_importances_ for est in model.estimators_ 
                    if hasattr(est, 'feature_importances_')]
            if imps:
                fold_imp = np.mean(imps, axis=0)
        
        feature_importances.append(fold_imp)
        
        print(f"\nCV={fold}{'-'*75}")
        print(f"Mean Metrics: R²={np.mean(fold_r2):.4f}, MAE={np.mean(fold_mae):.4f}, "
              f"MAPE={np.mean(fold_mape):.4f}%, RMSE={np.mean(fold_rmse):.4f}")
        
        for i, name in enumerate(target_names):
            print(f"  {name}: R²={fold_r2[i]:.4f}, MAE={fold_mae[i]:.4f}, "
                  f"MAPE={fold_mape[i]:.4f}%, RMSE={fold_rmse[i]:.4f}")
    
    print("\n" + "="*80)
    print("SUMMARY WITH 95% CONFIDENCE INTERVALS")
    print("="*80)
    
    for metric in ['r2', 'mae', 'mape', 'rmse']:
        mean, lower, upper = calculate_ci(metrics[metric])
        print(f"Mean {metric.upper()}: {mean:.4f} (95% CI: {lower:.4f}-{upper:.4f})")
    
    print("\nPER-TARGET METRICS:")
    for i, name in enumerate(target_names):
        print(f"\n{name}:")
        for metric in ['r2', 'mae', 'mape', 'rmse']:
            values = metrics[f'{metric}_target'][i]
            mean, lower, upper = calculate_ci(values)
            print(f"  {metric.upper()}: {mean:.4f} (95% CI: {lower:.4f}-{upper:.4f})")
    
    return metrics, feature_importances

In [44]:
# for GridSearch
rmse_scorer = make_scorer(rmse_score, greater_is_better=False)

In [45]:
# ExtraTreesRegressor
et_params = {
    'n_estimators': [50, 100],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}
et = ExtraTreesRegressor(random_state=42)
et_multi = MultiOutputRegressor(et)
et_grid = GridSearchCV(et_multi, param_grid={'estimator__' + k: v for k, v in et_params.items()},
                       scoring=rmse_scorer, n_jobs=-1, cv=5, verbose=1)
et_grid.fit(train_x, train_y)
print('Best Params:', et_grid.best_params_)
print('Best RMSE:', -et_grid.best_score_)
et_best = et_grid.best_estimator_

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Params: {'estimator__max_depth': None, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'estimator__n_estimators': 100}
Best RMSE: 1.9863297767779435


In [46]:
target_names = train_y.columns.tolist()  # target columns

In [47]:
# ExtraTreesRegressor
print("\n" + "="*80)
print("# EXTRA TREES REGRESSOR - DETAILED CV")
print("="*80)
et_metrics, et_importances = regression_cv(et_best, train_x, train_y, 
                                          target_names=target_names)


# EXTRA TREES REGRESSOR - DETAILED CV

CV=1---------------------------------------------------------------------------
Mean Metrics: R²=0.9842, MAE=1.0485, MAPE=1.8825%, RMSE=2.2224
  af_weight: R²=0.9813, MAE=0.9606, MAPE=1.5554%, RMSE=1.5755
  af_size: R²=0.9870, MAE=1.1364, MAPE=2.2096%, RMSE=2.8693

CV=2---------------------------------------------------------------------------
Mean Metrics: R²=0.9870, MAE=1.0657, MAPE=1.8800%, RMSE=1.7720
  af_weight: R²=0.9799, MAE=0.9850, MAPE=1.5862%, RMSE=1.5726
  af_size: R²=0.9942, MAE=1.1463, MAPE=2.1738%, RMSE=1.9713

CV=3---------------------------------------------------------------------------
Mean Metrics: R²=0.9847, MAE=1.0363, MAPE=1.8246%, RMSE=1.8316
  af_weight: R²=0.9748, MAE=1.0218, MAPE=1.6342%, RMSE=1.8132
  af_size: R²=0.9947, MAE=1.0508, MAPE=2.0150%, RMSE=1.8501

CV=4---------------------------------------------------------------------------
Mean Metrics: R²=0.9836, MAE=1.0735, MAPE=1.8894%, RMSE=2.0409
  af_weight: R²=0.9

In [48]:
# Optimal hyperparameter extraction function in GridSearchCV
def extract_best_params(grid_search_model):
    best_params = {}
    for key, value in grid_search_model.best_params_.items():
        clean_key = key.replace('estimator__', '')
        best_params[clean_key] = value
    return best_params

In [49]:
# CV function for regressorchain
def regression_cv_chain(model, X, y, cv=5, random_state=42, target_names=None):
    kf = KFold(n_splits=cv, shuffle=True, random_state=random_state)
    metrics = {
        'r2': [], 'mae': [], 'mape': [], 'rmse': [],
        'r2_target': [[] for _ in range(y.shape[1])],  # y.shape[1]으로 타겟 수 조정
        'mae_target': [[] for _ in range(y.shape[1])],
        'mape_target': [[] for _ in range(y.shape[1])],
        'rmse_target': [[] for _ in range(y.shape[1])]
    }
    
    if target_names is None:
        target_names = [f'Target_{i}' for i in range(y.shape[1])]
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        
        fold_r2, fold_mae, fold_mape, fold_rmse = [], [], [], []
        
        for i in range(y.shape[1]):  # y.shape[1]으로 타겟 수 정확히 지정
            y_val_i = y_val.iloc[:, i]
            y_pred_i = y_pred[:, i]
            
            r2_val = r2_score(y_val_i, y_pred_i)
            mae_val = mean_absolute_error(y_val_i, y_pred_i)
            mape_val = mape_score(y_val_i, y_pred_i)
            rmse_val = np.sqrt(mean_squared_error(y_val_i, y_pred_i))
            
            fold_r2.append(r2_val)
            fold_mae.append(mae_val)
            fold_mape.append(mape_val)
            fold_rmse.append(rmse_val)
            
            metrics['r2_target'][i].append(r2_val)
            metrics['mae_target'][i].append(mae_val)
            metrics['mape_target'][i].append(mape_val)
            metrics['rmse_target'][i].append(rmse_val)
        
        metrics['r2'].append(np.mean(fold_r2))
        metrics['mae'].append(np.mean(fold_mae))
        metrics['mape'].append(np.mean(fold_mape))
        metrics['rmse'].append(np.mean(fold_rmse))
        
        print(f"\nCV={fold}{'-'*75}")
        print(f"Mean Metrics: R²={np.mean(fold_r2):.4f}, MAE={np.mean(fold_mae):.4f}, "
              f"MAPE={np.mean(fold_mape):.4f}%, RMSE={np.mean(fold_rmse):.4f}")
        
        for i, name in enumerate(target_names):
            print(f"  {name}: R²={fold_r2[i]:.4f}, MAE={fold_mae[i]:.4f}, "
                  f"MAPE={fold_mape[i]:.4f}%, RMSE={fold_rmse[i]:.4f}")
    
    print("\n" + "="*80)
    print("SUMMARY WITH 95% CONFIDENCE INTERVALS")
    print("="*80)
    
    for metric in ['r2', 'mae', 'mape', 'rmse']:
        mean, lower, upper = calculate_ci(metrics[metric])
        print(f"Mean {metric.upper()}: {mean:.4f} (95% CI: {lower:.4f}-{upper:.4f})")
    
    print("\nPER-TARGET METRICS:")
    for i, name in enumerate(target_names):
        print(f"\n{name}:")
        for metric in ['r2', 'mae', 'mape', 'rmse']:
            values = metrics[f'{metric}_target'][i]
            mean, lower, upper = calculate_ci(values)
            print(f"  {metric.upper()}: {mean:.4f} (95% CI: {lower:.4f}-{upper:.4f})")
    
    return metrics

In [50]:
# ExtraTrees RegressorChain
print("\n" + "="*60)
print("# CHAINED: ExtraTrees")
print("="*60)

et_best_params = extract_best_params(et_grid)
print(f"Using ET parameters: {et_best_params}")

chained_et = RegressorChain(
    ExtraTreesRegressor(**et_best_params, random_state=42),
    order=[0, 1]  # order: af_weight -> af_size
)
# cv
chained_et_metrics = regression_cv_chain(chained_et, train_x, train_y, target_names=target_names)


# CHAINED: ExtraTrees
Using ET parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

CV=1---------------------------------------------------------------------------
Mean Metrics: R²=0.9845, MAE=1.0386, MAPE=1.8676%, RMSE=2.1795
  af_weight: R²=0.9813, MAE=0.9606, MAPE=1.5554%, RMSE=1.5755
  af_size: R²=0.9878, MAE=1.1165, MAPE=2.1798%, RMSE=2.7834

CV=2---------------------------------------------------------------------------
Mean Metrics: R²=0.9870, MAE=1.0572, MAPE=1.8749%, RMSE=1.7708
  af_weight: R²=0.9799, MAE=0.9850, MAPE=1.5862%, RMSE=1.5726
  af_size: R²=0.9942, MAE=1.1295, MAPE=2.1635%, RMSE=1.9691

CV=3---------------------------------------------------------------------------
Mean Metrics: R²=0.9848, MAE=1.0307, MAPE=1.8093%, RMSE=1.8249
  af_weight: R²=0.9748, MAE=1.0218, MAPE=1.6342%, RMSE=1.8132
  af_size: R²=0.9948, MAE=1.0396, MAPE=1.9844%, RMSE=1.8365

CV=4-----------------------------------------------------------------

Final model for PKL

In [51]:
feature_cols = list(train_x.columns)     

In [52]:
feature_cols

['Liposuction site',
 'Sex',
 'Age',
 'Height',
 'Weight',
 'Size',
 'TBW',
 'Body protein',
 'Body mineral',
 'FFM',
 'SMM',
 'BFM',
 'WHR',
 'BMI',
 'Liposuction type']

In [56]:
scaled_cols

['Body protein',
 'WHR',
 'SMM',
 'TBW',
 'Weight',
 'BFM',
 'Body mineral',
 'FFM',
 'Height',
 'Age',
 'Size',
 'BMI']

In [ ]:
order = list(getattr(chained_et, "order_", [1, 0])) 
target_names = list(train_y.columns)  # ['af_weight', 'af_size']

In [58]:
bundle = {
    "model": chained_et,   # Final model: ET
    "scaler": scaler,              # train_x[scaled_cols]에 fit된 StandardScaler
    "feature_cols": feature_cols,  # 입력 정렬용 (지금 순서 그대로)
    "scaled_cols": scaled_cols,    # 스케일 대상 컬럼(이 순서대로 transform할 것)
    "target_names": target_names,  # 출력 라벨
    "order": order               # 체인 학습 순서 기록 ([0,1]])
}

joblib.dump(bundle, "chained_et_final.pkl", compress=3)
print("save to pkl (overall ver; included scaler)")

save to pkl (overall ver; included scaler)


In [59]:
# inverse_transform
def inverse_scaler(df_scaled, scaler, scaled_cols):
    df_out = df_scaled.copy()
    df_out.loc[:, scaled_cols] = scaler.inverse_transform(df_out[scaled_cols])
    return df_out

In [60]:
test_x_inverse = inverse_scaler(test_x, scaler, scaled_cols)
test_x_inverse.to_csv("test_x_inverse.csv")
test_x_inverse

,Liposuction site,Sex,Age,Height,Weight,Size,TBW,Body protein,Body mineral,FFM,SMM,BFM,WHR,BMI,Liposuction type
424,2,2,28.0,152.0,50.100000,27.5,23.700000,6.300000,2.280000,32.300000,17.100000,17.800000,0.900000,21.680000,1
7004,6,2,52.0,162.0,56.800000,75.0,30.900000,8.200000,2.910000,42.000000,22.600000,14.800000,0.890000,21.640000,0
1600,1,2,42.0,165.0,63.130476,70.4,30.153462,8.050729,4.792275,41.167339,22.289032,21.962923,0.891351,23.136348,0
2827,2,2,34.0,158.0,57.400000,30.5,26.700000,7.200000,2.640000,36.500000,19.500000,20.900000,0.820000,22.990000,0
5965,1,2,31.0,158.0,60.900000,83.0,30.000000,8.100000,2.900000,41.000000,22.300000,19.900000,0.870000,24.400000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,1,2,41.0,158.0,67.400000,94.0,29.200000,7.900000,2.680000,39.800000,21.700000,27.600000,0.950000,27.000000,1
2074,7,2,35.0,167.0,67.400000,61.0,33.000000,8.700000,3.230000,44.900000,24.300000,22.500000,0.880000,24.170000,0
3741,2,2,34.0,156.0,60.200000,33.5,27.600000,7.400000,2.650000,37.600000,20.300000,22.600000,0.870000,24.740000,0
3749,2,2,28.0,162.0,71.500000,37.5,28.300000,7.400000,2.760000,38.500000,20.500000,33.000000,0.920000,27.240000,0


In [61]:
print("train_y.columns:", list(train_y.columns))
print("chained_et_reverse.order_:", chained_et.order_)


train_y.columns: ['af_weight', 'af_size']
chained_et_reverse.order_: [0, 1]


Abliation_Study

In [57]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd

In [54]:
ablation_order = ['Size','Liposuction site','Liposuction type','BMI']

In [55]:
ablation_sets = {'Full model': []}
removed = []

for feature in ablation_order:
    removed.append(feature)
    ablation_sets[f"Remove {' + '.join(removed)}"] = removed.copy()

# check
for k, v in ablation_sets.items():
    print(k, ":", v)

Full model : []
Remove Size : ['Size']
Remove Size + Liposuction site : ['Size', 'Liposuction site']
Remove Size + Liposuction site + Liposuction type : ['Size', 'Liposuction site', 'Liposuction type']
Remove Size + Liposuction site + Liposuction type + BMI : ['Size', 'Liposuction site', 'Liposuction type', 'BMI']


In [56]:
model_template = chained_et   # final_model

In [58]:
def evaluate_multioutput(y_true, y_pred, target_names):
    results = []

    for i, target in enumerate(target_names):
        r2 = r2_score(y_true.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true.iloc[:, i], y_pred[:, i]))
        mae = mean_absolute_error(y_true.iloc[:, i], y_pred[:, i])
        mape = mean_absolute_percentage_error(y_true.iloc[:, i], y_pred[:, i]) * 100

        results.append({
            'Target': target,
            'R²': r2,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE (%)': mape
        })

    return pd.DataFrame(results)

target_names = train_y.columns.tolist()
all_ablation_results = []

for setting_name, remove_features in ablation_sets.items():
    print("\n====================================")
    print(setting_name)
    print("Removed:", remove_features if remove_features else "None")
    print("====================================")

    selected_features = [col for col in feature_cols if col not in remove_features]

    X_train_ab = train_x[selected_features]
    X_test_ab = test_x[selected_features]

    # clone model so each setting is trained from scratch
    model = clone(model_template)

    model.fit(X_train_ab, train_y)
    pred = model.predict(X_test_ab)

    result_df = evaluate_multioutput(test_y, pred, target_names)

    result_df['Ablation setting'] = setting_name
    result_df['Removed features'] = ', '.join(remove_features) if remove_features else 'None'
    result_df['Number of features'] = len(selected_features)

    all_ablation_results.append(result_df)

ablation_results_df = pd.concat(all_ablation_results, ignore_index=True)

print("\nAblation results by target:")
print(ablation_results_df)


Full model
Removed: None

Remove Size
Removed: ['Size']

Remove Size + Liposuction site
Removed: ['Size', 'Liposuction site']

Remove Size + Liposuction site + Liposuction type
Removed: ['Size', 'Liposuction site', 'Liposuction type']

Remove Size + Liposuction site + Liposuction type + BMI
Removed: ['Size', 'Liposuction site', 'Liposuction type', 'BMI']

Ablation results by target:
      Target        R²       RMSE        MAE   MAPE (%)  \
0  af_weight  0.975320   1.707889   0.962307   1.547791   
1    af_size  0.987763   2.812319   1.130078   2.120636   
2  af_weight  0.975907   1.687464   0.921895   1.482056   
3    af_size  0.974643   4.048380   1.915872   3.407372   
4  af_weight  0.975580   1.698895   0.856296   1.375713   
5    af_size  0.015992  25.219281  18.814841  42.719207   
6  af_weight  0.972831   1.791944   0.884089   1.412884   
7    af_size  0.016938  25.207158  18.805048  42.758833   
8  af_weight  0.973365   1.774252   0.882054   1.412870   
9    af_size  0.018269 

In [59]:
ablation_summary = (
    ablation_results_df
    .groupby(['Ablation setting', 'Removed features', 'Number of features'])[['R²', 'RMSE', 'MAE', 'MAPE (%)']]
    .mean()
    .reset_index()
)

print("\nMean ablation summary:")
print(ablation_summary)


Mean ablation summary:
                                    Ablation setting  \
0                                         Full model   
1                                        Remove Size   
2                     Remove Size + Liposuction site   
3  Remove Size + Liposuction site + Liposuction type   
4  Remove Size + Liposuction site + Liposuction t...   

                                Removed features  Number of features  \
0                                           None                  15   
1                                           Size                  14   
2                         Size, Liposuction site                  13   
3       Size, Liposuction site, Liposuction type                  12   
4  Size, Liposuction site, Liposuction type, BMI                  11   

         R²       RMSE       MAE   MAPE (%)  
0  0.981542   2.260104  1.046193   1.834214  
1  0.975275   2.867922  1.418883   2.444714  
2  0.495786  13.459088  9.835569  22.047460  
3  0.494885  13.499551

In [60]:
save_path = r"/Users/selina/Desktop/365mc_re/0.code/3_analysis/ablation_results.csv"

ablation_results_df.to_csv(save_path.replace(".csv", "_by_target.csv"), index=False)
ablation_summary.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: /Users/selina/Desktop/365mc_re/0.code/3_analysis/ablation_results.csv
